In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install ftfy nltk gensim pandas numpy scikit-learn
print('Installation complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 28.4 MB/s eta 0:00:00
Installation complete


In [ ]:
!pip install vaderSentiment langdetect --quiet

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('✅ NLTK data downloaded')

✅ NLTK data downloaded


In [ ]:
BASE = '/content/drive/MyDrive/Text Miners/Actual Work'
print('Connected:', BASE)

Connected: /content/drive/MyDrive/Text Miners/Actual Work


In [ ]:
import pandas as pd
df = pd.read_csv(f'{BASE}/data/MergedCleaned.csv')
print('Shape:', df.shape)

Shape: (380505, 4)


In [ ]:
import sys
sys.path.append(f'{BASE}/pre-processing')
import preprocess

print('✅ preprocess.py imported')

✅ preprocess.py imported


In [ ]:
# This applies clean_text() to all reviews at once
df = preprocess.preprocess_dataframe(df, text_col='comments')

print('✅ Task 1 Complete: All reviews preprocessed')
print(f'Total reviews: {len(df)}')
print('\nExample:')
print(df[['comments', 'cleaned_text', 'tokens']].head(2))

✅ Task 1 Complete: All reviews preprocessed
Total reviews: 380505

Example:
                                            comments  \
0  We stayed in the apartment for a week and we e...   
1  My girlfriend and I recently stayed in Nuttee'...   

                                        cleaned_text  \
0  stayed apartment week enjoyed very much nuttee...   
1  girlfriend recently stayed nuttee condo month ...   

                                              tokens  
0  [stayed, apartment, week, enjoyed, very, much,...  
1  [girlfriend, recently, stayed, nuttee, condo, ...  


In [ ]:
# Extract the tokens column as a list of lists
sentences = df['tokens'].tolist()

print('✅ Task 2 Complete: Tokenized into list of word lists')
print(f'Total reviews: {len(sentences)}')
print(f'\nFirst review tokens: {sentences[0][:10]}...')  # Show first 10 words

✅ Task 2 Complete: Tokenized into list of word lists
Total reviews: 380505

First review tokens: ['stayed', 'apartment', 'week', 'enjoyed', 'very', 'much', 'nuttee', 'very', 'nice', 'host']...


In [ ]:
from gensim.models import Word2Vec

print('Training Word2Vec model...')
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    epochs=10,
    workers=4
)

print('✅ Task 3 Complete: Word2Vec model trained')
print(f'Vocabulary size: {len(w2v_model.wv)}')

Training Word2Vec model...
✅ Task 3 Complete: Word2Vec model trained
Vocabulary size: 28321


In [ ]:
# Test that similar words have similar vectors
test_words = ['clean', 'spotless', 'dirty', 'location', 'comfortable']

print('✅ Task 4: Word2Vec learns meaning from context\n')

for word in test_words:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        print(f'Words similar to "{word}":')
        for sim_word, score in similar:
            print(f'  - {sim_word}: {score:.3f}')
        print()

✅ Task 4: Word2Vec learns meaning from context

Words similar to "clean":
  - spotless: 0.679
  - neat: 0.631
  - stylish: 0.574
  - cozy: 0.564
  - spacious: 0.551

Words similar to "spotless":
  - immaculate: 0.803
  - spotlessly: 0.744
  - pristine: 0.730
  - impeccably: 0.716
  - stylish: 0.697

Words similar to "dirty":
  - gross: 0.828
  - dusty: 0.821
  - stained: 0.808
  - filthy: 0.798
  - stain: 0.787

Words similar to "location":
  - position: 0.661
  - spot: 0.487
  - localization: 0.484
  - area: 0.479
  - locale: 0.456

Words similar to "comfortable":
  - comfy: 0.871
  - cozy: 0.719
  - confortable: 0.716
  - cosy: 0.660
  - tidy: 0.624



In [ ]:
import os

# Ensure features directory exists
os.makedirs(f'{BASE}/features', exist_ok=True)

# Save the model
w2v_model.save(f'{BASE}/features/word2vec.model')

print('✅ Task 5 Complete: Model exported to /features/word2vec.model')

✅ Task 5 Complete: Model exported to /features/word2vec.model


In [ ]:
import numpy as np
import pickle

def get_review_embedding(tokens, model):
    """
    Average all word vectors in a review to get a single review vector.
    """
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        # Return zero vector if no words found in vocabulary
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

# Apply to all reviews
print('Creating review embeddings...')
df['w2v_embedding'] = df['tokens'].apply(
    lambda tokens: get_review_embedding(tokens, w2v_model)
)

# Convert to numpy array
embeddings_array = np.vstack(df['w2v_embedding'].values)

# Save as pickle file
with open(f'{BASE}/features/w2v_embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings_array, f)

print('✅ Task 6 Complete: Review embeddings created and exported')
print(f'   Saved {len(embeddings_array)} embeddings')
print(f'   Shape: {embeddings_array.shape}')
print(f'   File: /features/w2v_embeddings.pkl')

Creating review embeddings...
✅ Task 6 Complete: Review embeddings created and exported
   Saved 380505 embeddings
   Shape: (380505, 100)
   File: /features/w2v_embeddings.pkl


In [ ]:
import os

# Check that files exist
model_path = f'{BASE}/features/word2vec.model'
embeddings_path = f'{BASE}/features/w2v_embeddings.pkl'

print('Final verification:')
print(f'✅ word2vec.model exists: {os.path.exists(model_path)}')
print(f'✅ w2v_embeddings.pkl exists: {os.path.exists(embeddings_path)}')
print(f'\n🎉 All tasks complete!')

Final verification:
✅ word2vec.model exists: True
✅ w2v_embeddings.pkl exists: True

🎉 All tasks complete!
